In [92]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.append("..")
import pandas as pd
import numpy as np
from statsmodels import api as sm
from statsmodels.regression.linear_model import OLS
from src.plot import cumulative_returns, rolling_sharpe
from config import rf, gamma, method

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [93]:
BASE_DIR = Path().resolve().parent
portfolio_path = BASE_DIR / "results" / "portfolio"
metrics_path = BASE_DIR / "results" / "metrics"
prices_path = BASE_DIR / "results" / "prices"

portfolio_net_return = {}
portfolio_weights = {}
portfolio_turnover = {}
portfolio_sharpe_ratio_annual = {}
portfolio_sharpe_ratio_rolling = {}
portfolio_drawdown = {}
portfolio_informatio_ratio = {}
portfolio_effective_number = {}

for parameter in gamma:
    portfolio_net_return[parameter] = pd.read_parquet(portfolio_path / f"portfolio_net_returns_{parameter}.parquet").squeeze()
    portfolio_weights[parameter] = pd.read_parquet(portfolio_path / f"portfolio_weights_{parameter}.parquet")
    portfolio_turnover[parameter] = pd.read_parquet(metrics_path / f"portfolio_turnover_{parameter}.parquet").squeeze()
    portfolio_sharpe_ratio_annual[parameter] = pd.read_parquet(metrics_path / f"portfolio_sharpe_ratio_{parameter}_annual.parquet")
    portfolio_sharpe_ratio_rolling[parameter] = pd.read_parquet(metrics_path / f"portfolio_sharpe_ratio_{parameter}_rolling.parquet").squeeze()
    portfolio_drawdown[parameter] = pd.read_parquet(metrics_path / f"portfolio_drawdown_{parameter}.parquet").squeeze()
    portfolio_informatio_ratio[parameter] = pd.read_parquet(metrics_path / f"portfolio_information_ratio_{parameter}.parquet")
    portfolio_effective_number[parameter] = pd.read_parquet(metrics_path / f"portfolio_effective_number_{parameter}.parquet").squeeze()
    
equal_weight_net_return = pd.read_parquet(portfolio_path / f"equal_weight_returns.parquet").squeeze()
equal_weight_weights = pd.read_parquet(portfolio_path / f"equal_weight_weights.parquet")
equal_weight_turnover = pd.read_parquet(metrics_path / f"equal_weight_turnover.parquet").squeeze()
equal_weight_sharpe_ratio_annual = pd.read_parquet(metrics_path / f"equal_weight_sharpe_ratio_annual.parquet")
equal_weight_sharpe_ratio_rolling = pd.read_parquet(metrics_path / f"equal_weight_sharpe_ratio_rolling.parquet").squeeze()
equal_weight_drawdown = pd.read_parquet(metrics_path / f"equal_weight_drawdown.parquet").squeeze()
equal_weight_informatio_ratio = pd.read_parquet(metrics_path / f"equal_weight_informatio_ratio.parquet")
equal_weight_effective_number = pd.read_parquet(metrics_path / f"equal_weight_effective_number.parquet").squeeze()

market_portfolio_returns = pd.read_parquet(prices_path / "market_portfolio.parquet").squeeze()
market_portfolio_returns = market_portfolio_returns.loc[portfolio_net_return[1].index]
if method == "log":
    market_portfolio_returns = np.exp(market_portfolio_returns) - 1
    
market_portfolio_sharpe_ratio_annual = pd.read_parquet(metrics_path / f"market_portfolio_sharpe_ratio_annual.parquet")
market_portfolio_sharpe_ratio_rolling = pd.read_parquet(metrics_path / f"market_portfolio_sharpe_ratio_rolling.parquet").squeeze()
market_portfolio_drawdown = pd.read_parquet(metrics_path / f"market_portfolio_drawdown.parquet").squeeze()

In [94]:
x = market_portfolio_returns - rf
x = sm.add_constant(x)
alpha = {}

for parameter in gamma:
    y = portfolio_net_return[parameter] - rf
    alpha[parameter] = OLS(y, x).fit()
    
for parameter in gamma:
    print(alpha[parameter].summary())

                            OLS Regression Results                            
Dep. Variable:             Net Return   R-squared:                       0.465
Model:                            OLS   Adj. R-squared:                  0.465
Method:                 Least Squares   F-statistic:                     1310.
Date:                Sun, 13 Sep 2026   Prob (F-statistic):          7.10e-207
Time:                        00:05:56   Log-Likelihood:                 3973.7
No. Observations:                1508   AIC:                            -7943.
Df Residuals:                    1506   BIC:                            -7933.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0003      0.000      0.767      0.4

In [95]:
cumulative_returns(portfolio_net_return, market_portfolio_returns, equal_weight_net_return)
rolling_sharpe(portfolio_sharpe_ratio_rolling, equal_weight_sharpe_ratio_rolling, market_portfolio_sharpe_ratio_rolling)

In [96]:
for parameter in gamma:
    print(f"\n{'='*40}")
    print(f"Gamma = {parameter}")
    print(f"{'='*40}")

    print(f"Annualised Sharpe ratio : {portfolio_sharpe_ratio_annual[parameter].iloc[0, 0]:.3f}")
    print(f"Maximum drawdown : {portfolio_drawdown[parameter].min():.3%}")
    print(f"Information Ratio : {portfolio_informatio_ratio[parameter].iloc[0, 0]:.3f}")
    print(f"Annualised turnover : {portfolio_turnover[parameter].mean() * 252:.2%}")
    print(f"Annualised return : {portfolio_net_return[parameter].mean() * 252:.3%}")
    print(f"Annualised volatility : {portfolio_net_return[parameter].std() * np.sqrt(252):.3%}")
    print(f"Effective assets : {portfolio_effective_number[parameter].mean():.2f}")

print(f"\n{'='*40}")
print(f"Equal Weight")
print(f"{'='*40}")
print(f"Annualised Sharpe ratio : {equal_weight_sharpe_ratio_annual.iloc[0, 0]:.3f}")
print(f"Maximum drawdown : {equal_weight_drawdown.min():.3%}")
print(f"Information Ratio : {equal_weight_informatio_ratio.iloc[0, 0]:.3f}")
print(f"Annualised turnover : {equal_weight_turnover.mean() * 252:.3%}")
print(f"Annualised return : {equal_weight_net_return.mean() * 252:.3%}")
print(f"Annualised volatility : {equal_weight_net_return.std() * np.sqrt(252):.3%}")
print(f"Effective assets : {equal_weight_effective_number.mean():.2f}")

print(f"\n{'='*40}")
print(f"Benchmark")
print(f"{'='*40}")
print(f"Annualised Sharpe ratio : {market_portfolio_sharpe_ratio_annual.iloc[0, 0]:.3f}")
print(f"Maximum drawdown : {market_portfolio_drawdown.min():.3%}")
print(f"Annualised return : {market_portfolio_returns.mean() * 252:.3%}")
print(f"Annualised volatility : {market_portfolio_returns.std() * np.sqrt(252):.3%}")


Gamma = 1
Annualised Sharpe ratio : 0.820
Maximum drawdown : -44.374%
Information Ratio : 0.365
Annualised turnover : 310.91%
Annualised return : 34.576%
Annualised volatility : 37.677%
Effective assets : 3.48

Gamma = 5
Annualised Sharpe ratio : 0.672
Maximum drawdown : -44.378%
Information Ratio : 0.083
Annualised turnover : 313.75%
Annualised return : 26.549%
Annualised volatility : 34.052%
Effective assets : 3.59

Gamma = 10
Annualised Sharpe ratio : 0.616
Maximum drawdown : -44.679%
Information Ratio : -0.069
Annualised turnover : 307.95%
Annualised return : 22.937%
Annualised volatility : 31.268%
Effective assets : 3.82

Gamma = 25
Annualised Sharpe ratio : 0.458
Maximum drawdown : -44.643%
Information Ratio : -0.451
Annualised turnover : 284.30%
Annualised return : 15.671%
Annualised volatility : 26.178%
Effective assets : 4.80

Gamma = 50
Annualised Sharpe ratio : 0.437
Maximum drawdown : -39.595%
Information Ratio : -0.588
Annualised turnover : 241.32%
Annualised return : 13.